In [1]:
from langchain.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """将两个数字相加。"""
    print(f"--- 调用工具 'add_numbers' 计算 {a} + {b} ---") # 调试信息
    return a + b

In [2]:
# 直接调用工具函数
result = add_numbers.invoke({"a": 10, "b": 5})
print("计算结果:", result)

--- 调用工具 'add_numbers' 计算 10.0 + 5.0 ---
计算结果: 15.0


In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """将两个数字相加。"""
    print(f"--- 调用工具 'add_numbers' 计算 {a} + {b} ---")
    return a + b

os.chdir("D:/my-agent-project")
load_dotenv()
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0
)

tools = [add_numbers]
agent_executor = create_react_agent(llm, tools)

user_input = "请计算 256 加上 128 等于多少？"
for event in agent_executor.stream(
    {"messages": [("user", user_input)]},
    {"recursion_limit": 15}
):
    for value in event.values():
        value["messages"][-1].pretty_print()

C:\Users\limin\AppData\Local\Temp\ipykernel_1612\2656150827.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


================================== Ai Message ==================================

好的，我来计算 256 加上 128。
Tool Calls:
  add_numbers (call_00_mj3jElf6InYpLAAf04Za3546)
 Call ID: call_00_mj3jElf6InYpLAAf04Za3546
  Args:
    a: 256
    b: 128
--- 调用工具 'add_numbers' 计算 256.0 + 128.0 ---
================================= Tool Message =================================
Name: add_numbers

384.0
================================== Ai Message ==================================

256 加上 128 等于 **384**。


In [4]:
import requests

# 加载环境变量（包括 DEEPSEEK_API_KEY 和 AMAP_API_KEY）
load_dotenv()

# ================= 定义天气工具 =================
@tool
def get_weather_amap(city_name: str) -> str:
    """通过高德地图API获取中国城市的实时天气信息。"""
    print(f"--- 正在通过高德地图查询 {city_name} 的天气 ---")
    try:
        geo_url = "https://restapi.amap.com/v3/geocode/geo"
        geo_params = {
            "address": city_name,
            "key": os.getenv("AMAP_API_KEY")
        }
        geo_response = requests.get(geo_url, params=geo_params)
        geo_response.raise_for_status()
        geo_data = geo_response.json()
        if geo_data.get("status") != "1":
            return f"地理编码失败: {geo_data.get('info')}"
        geocodes = geo_data.get("geocodes")
        if not geocodes:
            return f"未找到城市: {city_name}"
        adcode = geocodes[0].get("adcode")
        city_display = geocodes[0].get("city") or geocodes[0].get("province", city_name)
        weather_url = "https://restapi.amap.com/v3/weather/weatherInfo"
        weather_params = {
            "city": adcode,
            "key": os.getenv("AMAP_API_KEY"),
            "extensions": "base"
        }
        weather_response = requests.get(weather_url, params=weather_params)
        weather_response.raise_for_status()
        weather_data = weather_response.json()
        if weather_data.get("status") != "1":
            return f"天气查询失败: {weather_data.get('info')}"
        lives = weather_data.get("lives")
        if not lives:
            return f"无天气数据: {city_display}"
        live = lives[0]
        return (f"城市: {city_display}\n"
                f"天气: {live['weather']}\n"
                f"温度: {live['temperature']}°C\n"
                f"湿度: {live['humidity']}%\n"
                f"风向: {live['winddirection']}\n"
                f"风力: {live['windpower']}级\n"
                f"更新时间: {live['reporttime']}")
    except Exception as e:
        return f"查询异常: {str(e)}"
# ========== 工具2：加法 ==========
@tool
def add_numbers(a: float, b: float) -> float:
    """将两个数字相加。"""
    print(f"--- 计算 {a} + {b} = {a+b} ---")
    return a + b

# ================= 初始化大模型 =================
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
    temperature=0
)

# ================= 创建 Agent =================
tools = [get_weather_amap,add_numbers]   # 将天气工具加入列表
agent_executor = create_react_agent(llm, tools)

# ================= 测试 Agent =================
user_input = "上海的温度加上20度等于多少？"
print(f"用户: {user_input}\n")
for event in agent_executor.stream(
    {"messages": [("user", user_input)]},
    {"recursion_limit": 15}
):
    for value in event.values():
        value["messages"][-1].pretty_print()

用户: 上海的温度加上20度等于多少？



C:\Users\limin\AppData\Local\Temp\ipykernel_1612\1709051170.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


================================== Ai Message ==================================

我来查一下上海的天气温度。
Tool Calls:
  get_weather_amap (call_00_XeZmMFPdvhvPnddCet3i6117)
 Call ID: call_00_XeZmMFPdvhvPnddCet3i6117
  Args:
    city_name: 上海
--- 正在通过高德地图查询 上海 的天气 ---
================================= Tool Message =================================
Name: get_weather_amap

城市: 上海市
天气: 阴
温度: 26°C
湿度: 60%
风向: 东北
风力: ≤3级
更新时间: 2026-05-22 14:01:49
================================== Ai Message ==================================

上海当前温度为 **26°C**。

那么，26°C + 20°C = **46°C**。

所以，上海的温度加上20度等于 **46°C**。
